# Lab 2.2 – ZooKeeper: Distributed Lock (Tugas Inti)

**Mata Kuliah:** Sistem Terdistribusi  
**Topik:** Koordinasi Terdistribusi — Distributed Lock menggunakan ZooKeeper & Kazoo

---

## Deskripsi Tugas

| No | Tugas | Deskripsi |
|----|-------|-----------|
| **1** | Amati Output | Jalankan kode dasar (3 worker), amati apakah hanya 1 worker di *Critical Section* (CR) |
| **2** | Ubah Worker | Tambah jumlah worker menjadi 5, apakah urutan selalu sama? Mengapa? |
| **3** | Simulasi Crash | Tambahkan `raise Exception` di dalam `with lock`, apa yang terjadi pada worker lain? |
| **4** | Manual Lock | Ganti `Lock` dengan implementasi manual menggunakan *ephemeral sequential node* |

> **Prasyarat:** Docker Desktop berjalan dan container ZooKeeper aktif (`zk-lab2` dari Lab 2.1)

---
## Konsep Dasar yang Perlu Dipahami

### Apa itu Critical Section (CR)?

> **Critical Section** adalah bagian kode yang hanya boleh dieksekusi oleh **satu proses/thread pada satu waktu** untuk mencegah *race condition* (kondisi balapan data).

**Analogi nyata:** Bayangkan sebuah toilet umum:
- Hanya **1 orang** yang bisa masuk dalam satu waktu (ada kuncinya)
- Yang lain **mengantri** di luar dan menunggu
- Setelah selesai, kunci dilepas → orang berikutnya bisa masuk

---

### Mengapa Butuh *Distributed* Lock?

Pada sistem terdistribusi, banyak proses berjalan di **mesin berbeda**. Lock Python biasa (`threading.Lock()`) **tidak cukup** karena hanya berlaku dalam satu proses di satu mesin.

**Solusi:** Gunakan ZooKeeper sebagai **koordinator terpusat** yang bisa diakses semua mesin di jaringan!

---

### Bagaimana Kazoo `Lock` Bekerja?

```
Worker-0 ──▶ buat node antrian di ZooKeeper (terdepan → dapat lock)
Worker-1 ──▶ buat node antrian (urutan ke-2 → tunggu Worker-0 selesai)
Worker-2 ──▶ buat node antrian (urutan ke-3 → tunggu Worker-1 selesai)

Worker-0 selesai ──▶ hapus nodenya ──▶ Worker-1 mendapat giliran
Worker-1 selesai ──▶ hapus nodenya ──▶ Worker-2 mendapat giliran
```

> **Keunggulan Ephemeral Node:** Jika Worker-0 tiba-tiba *crash*, ZooKeeper otomatis menghapus nodenya sehingga Worker-1 tetap bisa maju. Tidak ada *deadlock*!

In [2]:
# ============================================================
# SETUP – Verifikasi ZooKeeper berjalan dan kazoo terinstall
# ============================================================
import subprocess, time, sys

def run_cmd(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

# --- Install kazoo jika belum ada ---
try:
    import kazoo
    print(f"✔ kazoo sudah terinstall (versi {kazoo.__version__})")
except ImportError:
    print("⏳ Menginstall kazoo...")
    subprocess.run([sys.executable, "-m", "pip", "install", "kazoo", "-q"])
    import kazoo
    print(f"✔ kazoo berhasil diinstall (versi {kazoo.__version__})")

# --- Cek apakah container ZooKeeper sudah berjalan ---
r = run_cmd("docker ps --filter name=zk-lab2 --format {{.Names}}")
if "zk-lab2" in r.stdout:
    print("✔ Container ZooKeeper (zk-lab2) sudah berjalan")
else:
    print("▶ Container belum berjalan, mencoba menjalankan...")
    run_cmd("docker rm -f zk-lab2")
    r2 = run_cmd("docker run -d --name zk-lab2 -p 2181:2181 zookeeper:3.9")
    if r2.returncode == 0:
        print("⏳ Menunggu ZooKeeper siap (5 detik)...")
        time.sleep(5)
        print("✔ ZooKeeper berhasil dijalankan")
    else:
        print(f"❌ Gagal menjalankan container: {r2.stderr.strip()}")

# --- Uji koneksi ke ZooKeeper ---
from kazoo.client import KazooClient
zk_test = KazooClient(hosts="127.0.0.1:2181")
try:
    zk_test.start(timeout=10)
    print(f"✔ Koneksi ke ZooKeeper berhasil! (state={zk_test.state})")
    zk_test.stop()
except Exception as e:
    print(f"❌ Koneksi gagal: {e}")
    print("   → Pastikan Docker Desktop berjalan dan container zk-lab2 aktif")

✔ kazoo sudah terinstall (versi 2.11.0)
✔ Container ZooKeeper (zk-lab2) sudah berjalan
✔ Koneksi ke ZooKeeper berhasil! (state=CONNECTED)


---
## Kode Dasar – Penjelasan Baris per Baris

Berikut adalah kode dasar yang digunakan sebagai referensi untuk semua tugas:

```python
from kazoo.client import KazooClient   # Library untuk konek ke ZooKeeper
from kazoo.recipe.lock import Lock     # Resep built-in untuk Distributed Lock
import threading, time

def worker(worker_id):
    zk = KazooClient(hosts='localhost:2181')  # Buat koneksi ke ZooKeeper
    zk.start()                               # Mulai koneksi
    lock = Lock(zk, "/myapp/lock")           # Daftarkan lock di path /myapp/lock
    with lock:                               # <-- MINTA LOCK (blok sampai dapat)
        print(f"[Worker-{worker_id}] Masuk CR.")  # Baris ini hanya dijalankan
        time.sleep(2)                            # satu worker pada satu waktu!
        print(f"[Worker-{worker_id}] Selesai.")  #
    # <-- LOCK OTOMATIS DILEPAS saat keluar blok 'with'
    zk.stop()                                # Tutup koneksi
```

### Penjelasan Kunci:

| Baris | Penjelasan |
|-------|------------|
| `Lock(zk, "/myapp/lock")` | Membuat objek lock di path `/myapp/lock` di ZooKeeper |
| `with lock:` | Python otomatis **acquire** lock saat masuk blok dan **release** saat keluar |
| `time.sleep(2)` | Simulasi pekerjaan berat di dalam Critical Section |
| `zk.stop()` | Penting! Tutup koneksi agar ephemeral node dihapus |

> **Tip untuk Pemula:** Penggunaan `with lock:` jauh lebih aman dari `lock.acquire()` / `lock.release()` manual, karena lock dijamin dilepas meski ada error!

---
## Tugas 1 – Amati Output: Apakah Hanya 1 Worker di CR?

**Pertanyaan:** Jalankan kode di bawah dengan **3 worker**. Amati output-nya:
- Apakah ada 2 worker yang masuk CR secara bersamaan?
- Bagaimana urutan masuk dan keluarnya?

**Yang diharapkan terjadi:**
```
[Worker-0] Mencoba acquire lock...
[Worker-1] Mencoba acquire lock...
[Worker-2] Mencoba acquire lock...
[Worker-0] Lock diperoleh! Masuk CR.      <-- hanya Worker-0 di dalam
[Worker-0] Selesai, release lock.
[Worker-1] Lock diperoleh! Masuk CR.      <-- baru Worker-1 masuk
[Worker-1] Selesai, release lock.
[Worker-2] Lock diperoleh! Masuk CR.      <-- baru Worker-2 masuk
[Worker-2] Selesai, release lock.
Semua worker selesai!
```

In [3]:
# ============================================================
# TUGAS 1 – Jalankan 3 Worker, Amati Output
# ============================================================
from kazoo.client import KazooClient
from kazoo.recipe.lock import Lock
import threading, time

# Fungsi yang dijalankan setiap worker
def worker(worker_id):
    # Setiap worker membuat koneksi sendiri ke ZooKeeper
    zk = KazooClient(hosts='127.0.0.1:2181')
    zk.start()

    # Semua worker berebut lock yang SAMA di path /myapp/lock
    lock = Lock(zk, "/myapp/lock")

    print(f"[Worker-{worker_id}] Mencoba acquire lock...")
    with lock:
        # ============== CRITICAL SECTION ==============
        # Kode di sini HANYA dijalankan oleh 1 worker pada 1 waktu
        print(f"[Worker-{worker_id}] Lock diperoleh! Masuk CR.")
        time.sleep(2)   # simulasi pekerjaan berat (misal: akses database)
        print(f"[Worker-{worker_id}] Selesai, release lock.")
        # ==============================================
    # Lock otomatis dilepas di sini

    zk.stop()

# Buat 3 thread yang masing-masing menjalankan worker()
NUM_WORKERS = 3
print(f"=" * 50)
print(f"TUGAS 1: Menjalankan {NUM_WORKERS} worker secara bersamaan")
print(f"=" * 50)

threads = [threading.Thread(target=worker, args=(i,)) for i in range(NUM_WORKERS)]
for t in threads:
    t.start()   # Semua thread mulai hampir bersamaan
for t in threads:
    t.join()    # Tunggu semua thread selesai

print("Semua worker selesai!")

TUGAS 1: Menjalankan 3 worker secara bersamaan
[Worker-0] Mencoba acquire lock...
[Worker-1] Mencoba acquire lock...
[Worker-2] Mencoba acquire lock...
[Worker-0] Lock diperoleh! Masuk CR.
[Worker-0] Selesai, release lock.
[Worker-2] Lock diperoleh! Masuk CR.
[Worker-2] Selesai, release lock.
[Worker-1] Lock diperoleh! Masuk CR.
[Worker-1] Selesai, release lock.
Semua worker selesai!


### Jawaban Tugas 1

**Ya, hanya 1 worker yang berada di Critical Section (CR) pada satu waktu.**

Buktinya terlihat dari output:
- Setelah `[Worker-X] Masuk CR.` muncul, baru setelah `[Worker-X] Selesai, release lock.` muncul, barulah worker berikutnya bisa masuk
- Tidak pernah ada dua baris "Masuk CR" yang muncul berturutan tanpa "Selesai" di antaranya

**Mengapa bisa begitu?** Karena ZooKeeper memastikan hanya satu klien yang bisa memegang lock `"/myapp/lock"` pada satu waktu. Klien lain **diblokir** (tidak bisa lanjut) sampai lock dilepas.

```
Timeline:

t=0s   Worker-0: [masuk CR] ████████████████
       Worker-1: [tunggu]   ░░░░░░░░░░░░░░░░[masuk CR] ████████████████
       Worker-2: [tunggu]   ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░[masuk CR] ████
       ─────────────────────────────────────────────────────────────▶ waktu
```

---
## Tugas 2 – Ubah Jumlah Worker Menjadi 5

**Pertanyaan:**
1. Apakah **urutan** worker masuk CR selalu sama setiap kali dijalankan?
2. Mengapa bisa seperti itu?

**Petunjuk:** Coba jalankan sel di bawah beberapa kali dan perhatikan urutan `[Worker-X]`.

In [4]:
# ============================================================
# TUGAS 2 – Ubah Jumlah Worker Menjadi 5
# ============================================================
from kazoo.client import KazooClient
from kazoo.recipe.lock import Lock
import threading, time

def worker(worker_id):
    zk = KazooClient(hosts='127.0.0.1:2181')
    zk.start()
    lock = Lock(zk, "/myapp/lock")
    print(f"[Worker-{worker_id}] Mencoba acquire lock...")
    with lock:
        print(f"[Worker-{worker_id}] Lock diperoleh! Masuk CR.")
        time.sleep(1)   # Dikurangi jadi 1 detik agar tidak terlalu lama
        print(f"[Worker-{worker_id}] Selesai, release lock.")
    zk.stop()

# === PERUBAHAN: ganti menjadi 5 worker ===
NUM_WORKERS = 5
print(f"=" * 50)
print(f"TUGAS 2: Menjalankan {NUM_WORKERS} worker secara bersamaan")
print(f"Coba jalankan sel ini beberapa kali dan bandingkan urutannya!")
print(f"=" * 50)

threads = [threading.Thread(target=worker, args=(i,)) for i in range(NUM_WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Semua worker selesai!")

TUGAS 2: Menjalankan 5 worker secara bersamaan
Coba jalankan sel ini beberapa kali dan bandingkan urutannya!
[Worker-0] Mencoba acquire lock...
[Worker-3] Mencoba acquire lock...
[Worker-4] Mencoba acquire lock...
[Worker-2] Mencoba acquire lock...
[Worker-1] Mencoba acquire lock...
[Worker-0] Lock diperoleh! Masuk CR.
[Worker-0] Selesai, release lock.
[Worker-3] Lock diperoleh! Masuk CR.
[Worker-3] Selesai, release lock.
[Worker-4] Lock diperoleh! Masuk CR.
[Worker-4] Selesai, release lock.
[Worker-2] Lock diperoleh! Masuk CR.
[Worker-2] Selesai, release lock.
[Worker-1] Lock diperoleh! Masuk CR.
[Worker-1] Selesai, release lock.
Semua worker selesai!


### Jawaban Tugas 2

**Urutan worker masuk CR TIDAK selalu sama.** Contoh output yang mungkin berbeda:

```
Percobaan ke-1:  Worker-0 → Worker-3 → Worker-1 → Worker-4 → Worker-2
Percobaan ke-2:  Worker-2 → Worker-0 → Worker-4 → Worker-1 → Worker-3
```

**Mengapa tidak deterministik?**

Ada dua sumber ketidakpastian:

1. **Thread Scheduling OS:** Sistem operasi menentukan thread mana yang berjalan lebih dulu. Tidak ada jaminan urutan.
2. **Waktu Koneksi & Pembuatan Node:** Meskipun semua thread di-`start()` hampir bersamaan, waktu koneksi TCP ke ZooKeeper sedikit berbeda. Worker yang berhasil membuat node ZooKeeper lebih dulu akan mendapat antrian lebih awal.

**Yang SELALU terjamin:** Walaupun urutan berbeda, **hanya 1 worker yang berada di CR** pada satu waktu. Ini adalah jaminan *mutual exclusion* dari ZooKeeper.

> **Pelajaran penting:** Jangan pernah mengandalkan urutan eksekusi yang tetap dalam sistem terdistribusi. Desain sistem harus tetap benar apapun urutannya!

---
## Tugas 3 – Simulasi Worker Crash di Tengah Critical Section

**Skenario:** Worker-0 crash tepat di dalam Critical Section!

**Pertanyaan:** Apa yang terjadi pada worker-worker lain yang sedang mengantri?

**Cara mensimulasikan crash:** Tambahkan `raise Exception("Simulasi crash!")` di dalam blok `with lock:`.

**Prediksi:** Ada dua kemungkinan:
- ❌ **Deadlock:** Worker lain tersangkut selamanya karena lock tidak pernah dilepas
- ✅ **Lock dilepas otomatis:** Worker lain tetap bisa lanjut

Jalankan kode di bawah untuk mengetahui jawaban sebenarnya!

In [5]:
# ============================================================
# TUGAS 3 – Simulasi Worker Crash di Tengah Critical Section
# ============================================================
from kazoo.client import KazooClient
from kazoo.recipe.lock import Lock
import threading, time

def worker_with_crash(worker_id, should_crash=False):
    zk = KazooClient(hosts='127.0.0.1:2181')
    zk.start()
    lock = Lock(zk, "/myapp/lock")

    print(f"[Worker-{worker_id}] Mencoba acquire lock...")
    try:
        with lock:
            # ============== CRITICAL SECTION ==============
            print(f"[Worker-{worker_id}] Lock diperoleh! Masuk CR.")
            time.sleep(1)

            if should_crash:
                # Mensimulasikan crash/error di tengah Critical Section
                print(f"[Worker-{worker_id}] !!! CRASH/ERROR terjadi di tengah CR !!!")
                raise Exception("Simulasi crash di tengah Critical Section!")

            print(f"[Worker-{worker_id}] Selesai normal, release lock.")
            # ==============================================
        # Lock otomatis dilepas oleh 'with' meski ada exception

    except Exception as e:
        # Exception ditangkap DI LUAR blok 'with', artinya lock SUDAH dilepas
        print(f"[Worker-{worker_id}] Exception ditangkap: {e}")
        print(f"[Worker-{worker_id}] Lock telah dilepas otomatis oleh 'with'.")
    finally:
        zk.stop()

print("=" * 55)
print("TUGAS 3: Worker-0 akan crash di tengah Critical Section")
print("=" * 55)

# Worker-0 akan crash, Worker-1 dan Worker-2 normal
configs = [(0, True), (1, False), (2, False)]   # (worker_id, should_crash)
threads = [threading.Thread(target=worker_with_crash, args=(wid, crash))
           for wid, crash in configs]

for t in threads:
    t.start()
for t in threads:
    t.join()

print("\nSemua worker selesai! (tidak ada deadlock)")

TUGAS 3: Worker-0 akan crash di tengah Critical Section
[Worker-0] Mencoba acquire lock...
[Worker-2] Mencoba acquire lock...
[Worker-1] Mencoba acquire lock...
[Worker-2] Lock diperoleh! Masuk CR.
[Worker-2] Selesai normal, release lock.
[Worker-0] Lock diperoleh! Masuk CR.
[Worker-0] !!! CRASH/ERROR terjadi di tengah CR !!!
[Worker-0] Exception ditangkap: Simulasi crash di tengah Critical Section!
[Worker-0] Lock telah dilepas otomatis oleh 'with'.
[Worker-1] Lock diperoleh! Masuk CR.
[Worker-1] Selesai normal, release lock.

Semua worker selesai! (tidak ada deadlock)


### Jawaban Tugas 3

**Worker lain TETAP BISA LANJUT. Tidak terjadi deadlock!**

**Penjelasan:**

Ketika Worker-0 crash (melempar exception) di dalam blok `with lock:`, Python secara otomatis menjalankan kode *cleanup* dari `with` statement. Ini disebut **context manager protocol**.

```python
with lock:          # ← ini memanggil lock.__enter__()
    ...             # kode di dalam
    raise Exception # crash!
# ← lock.__exit__() dipanggil OTOMATIS meski ada exception
#   yang melakukan: lock.release()
```

**Alur lengkap:**

```
Worker-0: [masuk CR] → crash → lock.__exit__() → lock dilepas
                                                       ↓
                                              Worker-1 mendapat lock
Worker-1:                               [masuk CR] → selesai normal
                                                       ↓
                                              Worker-2 mendapat lock
Worker-2:                                              [masuk CR] → selesai normal
```

> **Ini adalah keunggulan besar ZooKeeper + Kazoo:** Bahkan jika proses benar-benar mati (bukan hanya exception), ZooKeeper akan mendeteksi koneksi putus dan otomatis menghapus ephemeral lock node, sehingga worker lain tetap bisa maju.

---
## Tugas 4 – Manual Lock dengan Ephemeral Sequential Node

Sejauh ini kita menggunakan `Lock` bawaan Kazoo. Sekarang kita akan **mengimplementasikan sendiri** mekanisme yang sama!

### Algoritma Manual Lock (Resep ZooKeeper)

Ini adalah algoritma resmi dari dokumentasi ZooKeeper:

**Langkah Acquire Lock:**
1. Buat **ephemeral sequential node** di `/lock/node-` → ZooKeeper memberinya nomor urut, misal `/lock/node-0000000003`
2. Ambil semua anak dari `/lock/` dan **urutkan secara leksikografis**
3. Jika node kita adalah yang **terkecil** (terdepan) → kita mendapat lock!
4. Jika tidak → **pasang watch** pada node tepat di depan kita, lalu **tunggu**
5. Ketika watch terpicu (node di depan kita dihapus) → **kembali ke langkah 2**

**Langkah Release Lock:**
- Cukup **hapus** ephemeral node kita

### Visualisasi Antrian

```
/lock/
  ├── node-0000000001  ← Worker-A (pemegang lock saat ini)
  ├── node-0000000002  ← Worker-B (menunggu node-0000000001)
  └── node-0000000003  ← Worker-C (menunggu node-0000000002)

Worker-A selesai → hapus node-0000000001
Worker-B mendapat notifikasi → cek lagi → node-0000000002 adalah terkecil → dapat lock!
```

In [6]:
# ============================================================
# TUGAS 4 – Implementasi Manual Lock dengan Ephemeral Sequential Node
# ============================================================
from kazoo.client import KazooClient
import threading, time

LOCK_BASE = "/lock"   # Path parent untuk semua node lock


def acquire_lock_manual(zk, my_node_path):
    """
    Algoritma ZooKeeper Manual Lock.
    Memblokir sampai worker ini mendapat giliran (menjadi node terkecil).
    """
    # Ambil hanya nama node (tanpa path parent)
    my_node = my_node_path.split("/")[-1]

    while True:
        # Langkah 2: Ambil dan urutkan semua node antrian
        children = sorted(zk.get_children(LOCK_BASE))

        # Langkah 3: Apakah kita adalah yang terdepan?
        if children[0] == my_node:
            # Ya! Kita mendapat lock.
            return

        # Langkah 4: Cari node tepat di depan kita
        my_index = children.index(my_node)
        prev_node_path = f"{LOCK_BASE}/{children[my_index - 1]}"

        # Pasang 'watch' dan tunggu node tersebut dihapus
        watch_event = threading.Event()

        def on_node_deleted(event):
            # Fungsi ini dipanggil ZooKeeper saat node yang di-watch berubah/dihapus
            watch_event.set()  # Beritahu thread kita bahwa ada perubahan

        # Langkah 5: Pasang watch (jika node masih ada)
        if zk.exists(prev_node_path, watch=on_node_deleted):
            watch_event.wait()  # Tunggu notifikasi dari ZooKeeper
        # Kembali ke langkah 2 (loop while)


def release_lock_manual(zk, my_node_path):
    """Lepas lock dengan menghapus node kita."""
    zk.delete(my_node_path)


def worker_manual(worker_id):
    zk = KazooClient(hosts='127.0.0.1:2181')
    zk.start()

    # Pastikan path parent /lock ada
    zk.ensure_path(LOCK_BASE)

    # Langkah 1: Buat ephemeral sequential node
    # ZooKeeper otomatis menambahkan nomor urut di belakang 'node-'
    # Contoh hasil: /lock/node-0000000001
    my_path = zk.create(
        f"{LOCK_BASE}/node-",
        ephemeral=True,    # Otomatis dihapus jika koneksi putus
        sequence=True      # ZooKeeper tambahkan nomor urut otomatis
    )
    my_node_name = my_path.split('/')[-1]
    print(f"[Worker-{worker_id}] Node dibuat: {my_node_name}, menunggu giliran...")

    # Coba acquire lock (blokir sampai dapat)
    acquire_lock_manual(zk, my_path)

    # ============== CRITICAL SECTION ==============
    print(f"[Worker-{worker_id}] Lock diperoleh! Masuk CR.")
    time.sleep(1.5)
    print(f"[Worker-{worker_id}] Selesai, keluar CR.")
    # ==============================================

    # Release lock: hapus node kita
    release_lock_manual(zk, my_path)
    print(f"[Worker-{worker_id}] Lock dilepas (node {my_node_name} dihapus).")

    zk.stop()


# --- Bersihkan node lama dari percobaan sebelumnya ---
zk_clean = KazooClient(hosts='127.0.0.1:2181')
zk_clean.start()
if zk_clean.exists(LOCK_BASE):
    zk_clean.delete(LOCK_BASE, recursive=True)
zk_clean.stop()

# --- Jalankan 3 worker menggunakan manual lock ---
NUM_WORKERS = 3
print("=" * 55)
print("TUGAS 4: Manual Lock (Ephemeral Sequential Node)")
print("=" * 55)

threads = [threading.Thread(target=worker_manual, args=(i,)) for i in range(NUM_WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Semua worker selesai!")

TUGAS 4: Manual Lock (Ephemeral Sequential Node)
[Worker-0] Node dibuat: node-0000000000, menunggu giliran...
[Worker-1] Node dibuat: node-0000000001, menunggu giliran...
[Worker-0] Lock diperoleh! Masuk CR.
[Worker-2] Node dibuat: node-0000000002, menunggu giliran...
[Worker-0] Selesai, keluar CR.
[Worker-0] Lock dilepas (node node-0000000000 dihapus).
[Worker-1] Lock diperoleh! Masuk CR.
[Worker-1] Selesai, keluar CR.
[Worker-1] Lock dilepas (node node-0000000001 dihapus).
[Worker-2] Lock diperoleh! Masuk CR.
[Worker-2] Selesai, keluar CR.
[Worker-2] Lock dilepas (node node-0000000002 dihapus).
Semua worker selesai!


### Jawaban Tugas 4

**Hasil implementasi manual ini identik dengan `kazoo.recipe.lock.Lock` — hanya 1 worker di CR pada satu waktu!**

### Perbandingan: `Lock` Bawaan vs. Manual

| Aspek | `kazoo.recipe.lock.Lock` | Implementasi Manual |
|-------|--------------------------|---------------------|
| Kemudahan | ✅ Sangat mudah, cukup `with lock:` | ❌ Perlu menulis algoritma sendiri |
| Transparansi | ❌ Mekanisme tersembunyi di dalam library | ✅ Kita tahu persis cara kerjanya |
| Kontrol | ❌ Terbatas pada fitur bawaan | ✅ Bisa dikustomisasi (timeout, dll.) |
| Bug-prone | ✅ Sudah teruji | ❌ Rentan bug jika implementasi keliru |

### Mengapa Ephemeral Sequential Node?

- **Ephemeral:** Jika worker crash, ZooKeeper otomatis hapus nodenya → tidak ada deadlock
- **Sequential:** ZooKeeper menjamin nomor urut selalu bertambah dan unik → urutan antrian terjamin adil (FIFO)
- **Watch pada node sebelumnya** (bukan semua node): Menghindari *herd effect* — jika satu lock dilepas, hanya 1 worker yang terbangun, bukan semua!

```
Herd Effect (BURUK):          Tanpa Herd Effect (BAIK):
Worker-A selesai               Worker-A selesai
  ↓ notifikasi ke SEMUA          ↓ notifikasi HANYA ke Worker-B
Worker-B, C, D, E terbangun    Worker-B terbangun → dapat lock
semua berebut → 4 losers       Worker-C, D, E tetap tidur
```

---
## Kesimpulan Lab 2.2

| Tugas | Temuan Utama |
|-------|--------------|
| **Tugas 1** | ZooKeeper Lock menjamin **mutual exclusion**: hanya 1 worker di CR pada satu waktu |
| **Tugas 2** | Urutan masuk CR **tidak deterministik** karena bergantung pada thread scheduling OS dan waktu koneksi TCP |
| **Tugas 3** | Lock **otomatis dilepas** meski worker crash, berkat `with` statement (context manager) dan ephemeral node ZooKeeper |
| **Tugas 4** | Di balik `kazoo.recipe.lock.Lock`, terdapat algoritma ephemeral sequential node yang efisien dan bebas herd effect |

### Kapan Menggunakan Distributed Lock?

- Koordinasi akses ke resource bersama (file, database, API dengan rate limit)
- Leader Election: menentukan siapa yang menjadi "koordinator" di antara banyak node
- Mencegah double processing pada job scheduler terdistribusi

### Keterbatasan Distributed Lock

- Menambah **latency** karena perlu round-trip ke ZooKeeper
- ZooKeeper menjadi **single point of failure** (meski bisa di-cluster)
- Pada kasus *network partition*, bisa terjadi dua node sama-sama mengklaim sebagai pemegang lock (masalah lanjutan: *fencing token*)

> **Referensi lanjutan:** Lihat materi tentang etcd (Raft-based KV Store) sebagai alternatif modern ZooKeeper untuk distributed coordination.